# Week 2 — Data-analytic thinking in Python
**AI in Business · MSc Business Analytics and AI**

| Part | Topic | Business question |
|---|---|---|
| A | NumPy: the dot product as a **weighted score** | How do we rank 1,000 customers with one line of code? |
| B | pandas + Matplotlib on a real churn dataset | Which 200 customers should we call this week? |

Reading for this week: Provost & Fawcett, *Data Science for Business*, chapters 1–2.

Cells marked **Exercise** are for you to complete, some in class and the rest is homework

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# We will work with pandas 3+ (the latest and coolest)
print("NumPy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# same "random" numbers for everyone
rng = np.random.default_rng(42)

# We can set some plotting parameters too
plt.rcParams.update({"figure.figsize": (7, 4), 
                     "axes.spines.top": False,
                     "axes.spines.right": False, 
                     "axes.grid": True,
                     "grid.alpha": 0.3})
# Set colors too: stayed / churned, used in every plot
BLUE, ORANGE = "#2a78d6", "#eb6834"

# NumPy: The workhorse behind numerical analysis and ML in python

## A1. Why arrays? Doing the same thing to many numbers at once

A Python list is a container. A NumPy **array** is a column of numbers you can do arithmetic on directly.

In [ ]:
prices_list = [10, 20, 30]
prices_list

In [ ]:
# a list repeats itself
print(prices_list * 2)    

In [ ]:
# To multiply every number bu a scalar you need
[i * 2 for i in prices_list]

In [ ]:
# Numpy and its vectorized actions
prices = np.array([10, 20, 30])
prices

In [ ]:
# an array multiplies every element
print(prices * 2)
# Add 20% VAT to every price
print(prices * 1.2)

In [ ]:
# Lets geenrate a numpy array with 1M prices
big = rng.uniform(0, 100, size=1_000_000)

# Now lets measure the time it takes a list to multiply each item by the VAT
t0 = time.perf_counter()
with_vat_loop = [p * 1.2 for p in big]
t1 = time.perf_counter()

# Lets do the same with numpy
with_vat_np = big * 1.2
t2 = time.perf_counter()

# Now we print the output
print(f"Python loop: {t1 - t0:.3f}s")
print(f"NumPy: {t2 - t1:.4f}s (about {(t1 - t0) / (t2 - t1):.0f}x faster)")

## A2. The dot product is a weighted score

A marketing team ranks sales leads with a simple rule:

> **score = 2 × (website visits) + 1 × (emails opened) + 0.5 × (years as customer)**

Each customer is a list of numbers (a **vector**). The rule is another vector, the **weights**.

| Customer | Visits | Emails opened | Years |
|---|---|---|---|
| Arta | 5 | 3 | 2 |
| Besnik | 1 | 8 | 10 |
| Drita | 7 | 0 | 1 |

For Arta: 2·5 + 1·3 + 0.5·2 = 10 + 3 + 1 = **14**.

Multiply matching entries, then add them up. That is all a **dot product** is:

$$ \mathbf{w}\cdot\mathbf{x} \;=\; w_1x_1 + w_2x_2 + \dots + w_nx_n \;=\; \sum_{i=1}^{n} w_i x_i $$

### Exercise 1
On paper, compute the scores of Besnik and Drita. I know this is boring, but understanding math is essential in analytics

In [ ]:
# Lets define a vector of weights: 3 features, thus vector length is 3
w = np.array([2, 1, 0.5])

# Pick Arta as an example
arta = np.array([5, 3, 2])

# print("Arta by hand :", 2*5 + 1*3 + 0.5*2)
print("Arta, np.dot :", np.dot(w, arta))
print("Arta, w @ x  :", w @ arta)     # @ is the dot-product operator

# Your turn: create besnik and drita and print their scores
# besnik = ...
# drita = ...

**Weights can be negative.** Add a fourth feature, *number of complaints*, with weight −3. A complaint now *lowers* the score.
Discuss: who decides these weights? (Next weeks: we let the data decide. A linear or logistic regression *is* a dot product whose weights are learned.)

In [ ]:
w4 = np.array([2, 1, 0.5, -3])
arta4 = np.array([5, 3, 2, 2])        # Arta complained twice
print("Arta with complaints:", w4 @ arta4)

## A3. Scoring everyone at once: matrix × vector

Stack the customers as rows of a table (a **matrix** `X`). Then `X @ w` takes the dot product of **every row** with `w`: one score per customer.

In [ ]:
X = np.array([[5, 3, 2],     # Arta
              [1, 8, 10],    # Besnik
              [7, 0, 1]])    # Drita
names = np.array(["Arta", "Besnik", "Drita"])

scores = X @ w
print(scores)
print("Ranking, best first:", names[np.argsort(-scores)])

### Simulated Scnario

Now a realistic size: **1,000 customers** (simulated data).

In [ ]:
# Let's soft code (remember python)
n = 1000
# website visits last month
visits = rng.poisson(4, n)
# emails opened
emails = rng.poisson(3, n)
# years as customer
years = rng.integers(0, 15, n)
# monthly income, EUR
income = rng.normal(900, 300, n).clip(300)

X_big = np.column_stack([visits, emails, years])
print("X_big has shape", X_big.shape, "(rows = customers, columns = features)")
print(X_big[:5])

### Exercise 2
1. Compute `scores_big`: the score of all 1,000 customers with one line.
2. Print the indices of the 10 best customers (hint: `np.argsort`).
3. What share of customers have a score above 15? (hint: `(scores_big > 15).mean()`)

In [ ]:
scores_big = X_big @ w
top10 = np.argsort(-scores_big)[:10]
print("Top 10 customers:", top10)
print("Their scores:   ", scores_big[top10])
print(f"Share with score > 15: {(scores_big > 15).mean():.1%}")

## A4. The units problem: why we standardize

Add **income (EUR)** to the score with a small weight of 0.05. Income is in the hundreds, the other features are single digits, so the column with the biggest numbers quietly takes over the ranking.

The fix is to put every feature on the same scale with a **z-score**:

$$ z = \frac{x - \text{mean}}{\text{standard deviation}} $$

A z-score of +1 means "one standard deviation above the average customer", whatever the original unit.

In [ ]:
X_inc = np.column_stack([visits, emails, years, income])

# We change the weight vector
w_inc = np.array([2, 1, 0.5, 0.05])

print("Column means:", X_inc.mean(axis=0).round(1))
print("Column std  :", X_inc.std(axis=0).round(1))

In [ ]:
raw_scores = X_inc @ w_inc
contrib = X_inc * w_inc
print("Average contribution of each feature:", contrib.mean(axis=0).round(1))

In [ ]:
# Now Standardize: We can write the formula above as such
Z = (X_inc - X_inc.mean(axis=0)) / X_inc.std(axis=0)
print("Means:", Z.mean(axis=0).round(3))
print("Stds :", Z.std(axis=0).round(3))

z_scores = Z @ np.array([2, 1, 0.5, 1])
top_raw = set(np.argsort(-raw_scores)[:100])
top_std = set(np.argsort(-z_scores)[:100])
print("Customers in both top-100 lists:", len(top_raw & top_std))

## A5. Look at the scores

A histogram shows how scores are spread. The dashed line marks the cut-off for the top 10%.

In [ ]:
scores_big = X_big @ w
cut = np.percentile(scores_big, 90)

# Plotit
fig, ax = plt.subplots()
ax.hist(scores_big, bins=30, color=BLUE, edgecolor="white")
ax.axvline(cut, color="black", linestyle="--", linewidth=1.5)
ax.text(cut + 0.3, ax.get_ylim()[1] * 0.9, f"top 10% (score ≥ {cut:.1f})")
ax.set_xlabel("Lead score")
ax.set_ylabel("Number of customers")
ax.set_title("Lead scores of 1,000 customers")
plt.show()

> **Take-away for Part A.** A score is a dot product. A model that scores customers (regression, logistic regression, the last layer of a neural network) computes `X @ w`. What changes from model to model is **how the weights are chosen**.

---
# Part B — pandas: a real churn problem

Provost & Fawcett's chapter 1 opens with *MegaTelCo*, a phone company losing customers. Here is a real version: the IBM **Telco Customer Churn** sample, about 7,000 customers of a telecom company. `Churn = Yes` means the customer left last month.

## B1. Load and inspect

In [ ]:
from pathlib import Path

LOCAL = Path("telco_churn.csv")
URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")

df = pd.read_csv(LOCAL if LOCAL.exists() else URL)
print(df.shape)
df.head()

In [ ]:
# A quick inspection: Nothing is missing, but "TotalCharges" charges is not numeric as as expected
df.info()

## B2. Data quality: always check before you analyse

`TotalCharges` should be a number, but `info()` shows it is stored as text (`object` in pandas 2, `str` in pandas 3). Let's find out why.

In [ ]:
# Let's convert TotalCharges to numeric
df["TotalCharges_num"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Find cases where conversion did not happen
bad = df[df["TotalCharges_num"].isna()]

# Show the cases
print(len(bad), "rows could not be converted")
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

### Exercise 4
Look at the `tenure` of these rows. What do they have in common, and what is the most sensible value for their `TotalCharges`? Fix the column, then drop the helper column.

In [ ]:
# All have tenure = 0: brand-new customers who have not been billed yet, lets make it 0
# Pandas will replace all NAs by 0
df["TotalCharges"] = df["TotalCharges_num"].fillna(0)
df = df.drop(columns="TotalCharges_num")

# Always check
print(df["TotalCharges"].dtype, "| missing:", df["TotalCharges"].isna().sum())

## B3. The target: how many customers leave?

In [ ]:
# We can have a simple check
df["Churn"].value_counts()

In [ ]:
# Turn to probabilities
df["Churn"].value_counts(normalize=True)

In [ ]:
# Or turn them into integers (Yes = 1, No = 0)
df["churned"] = (df["Churn"] == "Yes").astype(int)
base_rate = df["churned"].mean()
print(f"Base churn rate: {base_rate:.1%}  ({df['churned'].sum():,} of {len(df):,} customers)")

In [ ]:
# Now the data looks like this
df.head()

## B4. Who churns? `groupby`

The mean of a 0/1 column is a rate, so `groupby(...)["churned"].mean()` gives the churn rate per group.

In [ ]:
df["Contract"].value_counts()

In [ ]:
# One of the most powerful function of pandas
by_contract = (df.groupby("Contract")["churned"]
                 .agg(customers="size", churn_rate="mean")
                 .sort_values("churn_rate", ascending=False))
by_contract.head()

In [ ]:
by_contract["lift_vs_base"] = by_contract["churn_rate"] / base_rate
by_contract.round(3)

**Lift** = group churn rate ÷ base rate. A lift of 1.6 means "60% more likely to churn than the average customer".

### Exercise 5
Make the same table for `PaymentMethod` and for `InternetService`. Which single group has the highest churn rate? Is it a big group or a small one?

In [ ]:
# Split the for loop into two
for col in ["PaymentMethod", "InternetService"]:
    t = (df.groupby(col)["churned"]
           .agg(customers="size", churn_rate="mean")
           .sort_values("churn_rate", ascending=False))
    t["lift_vs_base"] = t["churn_rate"] / base_rate
    print(t.round(3), "\n")

## B5. Plots that answer a question

**Plot 1: churn rate by contract type**, with the base rate for reference.

In [ ]:
fig, ax = plt.subplots()
rates = by_contract["churn_rate"].sort_values()
bars = ax.barh(rates.index, rates.values, color=BLUE)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in rates.values], padding=3)
ax.axvline(base_rate, color="black", linestyle="--", linewidth=1.5)
ax.text(base_rate + 0.005, -0.45, f"base rate {base_rate:.1%}", fontsize=9)
ax.set_xlabel("Churn rate")
ax.set_title("Month-to-month customers churn the most")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.show()

**Plot 2: churn rate by tenure.** `pd.cut` turns months of tenure into bands.

In [ ]:
bins = [0, 6, 12, 24, 48, 72]
labels = ["0–6", "7–12", "13–24", "25–48", "49–72"]
df["tenure_band"] = pd.cut(df["tenure"], bins=bins, labels=labels, include_lowest=True)

by_tenure = df.groupby("tenure_band", observed=True)["churned"].mean()

fig, ax = plt.subplots()
bars = ax.bar(by_tenure.index.astype(str), by_tenure.values, color=BLUE)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in by_tenure.values], padding=3)
ax.axhline(base_rate, color="red", linestyle="--", linewidth=1.5)
ax.set_ylim(0, by_tenure.max() * 1.15)
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Churn rate")
ax.set_title("New customers are the most likely to leave")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.show()

**Plot 3: monthly charges of customers who stayed vs. left.**

In [ ]:
fig, ax = plt.subplots()
for label, color in [("No", BLUE), ("Yes", ORANGE)]:
    ax.hist(df.loc[df["Churn"] == label, "MonthlyCharges"], bins=30,
            alpha=0.6, color=color, density=True,
            label="Stayed" if label == "No" else "Churned")
ax.set_xlabel("Monthly charges (USD)")
ax.set_ylabel("Share of customers (density)")
ax.set_title("Churners pay more per month")
ax.legend()
plt.show()

**Plot 4: two variables at once.** A pivot table of churn rate by contract × internet service, shown as a heatmap.

In [ ]:
pivot = df.pivot_table(index="Contract", columns="InternetService",
                       values="churned", aggfunc="mean")
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap="Blues", vmin=0, vmax=pivot.values.max())
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.grid(False)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v:.0%}", ha="center", va="center",
                color="white" if v > pivot.values.max() * 0.6 else "black")
ax.set_title("Churn rate: contract × internet service")
plt.colorbar(im, ax=ax, format=plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.show()

## B6. The challenge: you can call 200 customers this week

The retention team has time for **200 calls**. Build a **hand-made risk score** with what you learned in Part A, and pick the 200 highest scores.

1. Turn the risk factors you found into 0/1 columns (e.g. month-to-month contract, fiber optic, electronic check, tenure ≤ 12 months, senior citizen).
2. Choose a weight for each one.
3. Score every customer with `F @ weights`, take the top 200, and compute their churn rate and **lift**.
4. Compare with 200 customers picked at random.

Report your lift to the class. Highest lift wins.

### Exercise 6

In [ ]:
F = pd.DataFrame({
    "month_to_month": (df["Contract"] == "Month-to-month").astype(int),
    # add more 0/1 columns here
})
weights = np.array([1])          # one weight per column of F

risk = F.to_numpy() @ weights
top200 = np.argsort(-risk, kind="stable")[:200]
# compute the churn rate and lift of top200

### Discussion (5 minutes)
- Your top 200 churn far more than average. **Is calling them the right decision?** What does a call cost, and what is a retained customer worth?
- Some of them will leave whatever you do; some would have stayed anyway. A churn score says **who will leave**, not **whose mind a call can change**. (Remember the "persuadables" and "sleeping dogs" from week 1.)
- Next week: Provost & Fawcett ch. 7–8. We turn this into an **expected-value** decision and choose how many to call by profit, not by gut feeling.

---
# Exit ticket (5 minutes, on paper)

**Q1.** Compute the dot product of **w** = (2, −1, 3) and **x** = (4, 5, 1).

**Q2.** A supermarket asks: *"Which products do customers buy together?"* Which type of data-mining task is this (Provost & Fawcett ch. 2), and is it supervised or unsupervised?

**Q3.** A bank's churn table:

| Card type | Customers | Churn rate |
|---|---|---|
| Basic | 6,000 | 12% |
| Gold | 3,000 | 6% |
| Student | 1,000 | 30% |

(a) What is the bank's overall churn rate? (b) What is the lift of the Student group? (c) Which group loses the **most customers** in absolute numbers?